In [3]:


import pandas as pd
import numpy as np


df = pd.read_csv("all_predictions_logreg_model1.csv")


Y_COL       = "TARGET"
C_COL       = "predicted"
PROB_COL    = "pred_proba"
POS_LABEL   = 1
POS_PRED    = 1

GROUP_COLS = [
    "NAME_EDUCATION_TYPE_Higher.education",
    "NAME_EDUCATION_TYPE_Secondary.secondary.special",
    "NAME_INCOME_TYPE_Working",
    "NAME_INCOME_TYPE_Pensioner",
    "REGION_RATING",
]
GROUP_COLS = [c for c in GROUP_COLS if c in df.columns]

def _to_bin(x, val):
    if pd.api.types.is_numeric_dtype(x):
        return (x == val).astype(int)
    return (x.astype(str) == str(val)).astype(int)

y  = _to_bin(df[Y_COL], POS_LABEL)
c1 = _to_bin(df[C_COL], POS_PRED)  # this is C=1 in notes

def rate(mask):
    m = mask.astype(bool).values
    return m.mean() if m.size else np.nan

def metric_table_for_group(gcol):
    ser = df[gcol]
    # Build rows
    rows = []
    for lev in pd.unique(ser.dropna()):
        mask_g = (ser == lev)
        n = int(mask_g.sum())
        if n == 0:
            continue
        sel = rate(c1[mask_g] == 1)                           # P(C=1 | A=g)
        mask_pos = (y == 1) & mask_g
        tpr = rate(c1[mask_pos] == 1) if mask_pos.sum() else np.nan    # P(C=1 | Y=1, A=g)
        mask_neg = (y == 0) & mask_g
        fpr = rate(c1[mask_neg] == 1) if mask_neg.sum() else np.nan    # P(C=1 | Y=0, A=g)
        mask_pred_pos = (c1 == 1) & mask_g
        ppv = rate(y[mask_pred_pos] == 1) if mask_pred_pos.sum() else np.nan # P(Y=1 | C=1, A=g)
        mask_pred_neg = (c1 == 0) & mask_g
        npv = rate(y[mask_pred_neg] == 0) if mask_pred_neg.sum() else np.nan # P(Y=0 | C=0, A=g)
        rows.append({
            "group_variable": gcol,
            "group_value": str(lev),
            "n": n,
            "selection_rate": sel,
            "equal_opportunity_recall_pos": tpr,
            "equal_odds_fpr": fpr,
            "ppv": ppv,
            "npv": npv
        })
    return pd.DataFrame(rows)

by_group_frames = [metric_table_for_group(gc) for gc in GROUP_COLS]
by_group = pd.concat(by_group_frames, ignore_index=True) if by_group_frames else pd.DataFrame()

print(by_group)



                          group_variable  group_value      n  selection_rate  \
0   NAME_EDUCATION_TYPE_Higher.education            0  46485        0.657416   
1   NAME_EDUCATION_TYPE_Higher.education            1  14883        0.339918   
2               NAME_INCOME_TYPE_Working            1  31961        0.700635   
3               NAME_INCOME_TYPE_Working            0  29407        0.449757   
4             NAME_INCOME_TYPE_Pensioner            0  50457        0.622986   
5             NAME_INCOME_TYPE_Pensioner            1  10911        0.383558   
6                          REGION_RATING  -0.08417231  45352        0.582642   
7                          REGION_RATING    -2.085497   6378        0.280182   
8                          REGION_RATING    1.9171524   8658        0.783784   
9                          REGION_RATING      0.91649    733        0.686221   
10                         REGION_RATING   -1.0848347    247        0.481781   

    equal_opportunity_recall_pos  equal